# Taxi Zone & Corridor Clustering
This notebook groups pickup zones and routes into clusters using KMeans.
It assigns labels like "High Demand Hub", "Slow / Congested", etc.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

data_folder = "../data/"
models_folder = "../models_saved/"
os.makedirs(models_folder, exist_ok=True)

print("Loading metrics files...")
zone_metrics = pd.read_parquet(data_folder + "zone_metrics.parquet", engine='fastparquet')
corridor_metrics = pd.read_parquet(data_folder + "corridor_metrics.parquet", engine='fastparquet')

print("Zone data shape:", zone_metrics.shape)
print("Corridor data shape:", corridor_metrics.shape)
print("Zone columns:", list(zone_metrics.columns))
print("Corridor columns:", list(corridor_metrics.columns))

Loading metrics files...
Zone data shape: (4524, 7)
Corridor data shape: (72447, 6)
Zone columns: ['pickup_zone', 'pickup_borough', 'pickup_hour', 'avg_duration', 'avg_price', 'avg_speed', 'trip_count']
Corridor columns: ['route', 'pickup_hour', 'avg_duration', 'avg_price', 'avg_speed', 'trip_count']


In [2]:
# Create a summary of each zone (averaging across all hours)
zone_summary = zone_metrics.groupby(['pickup_zone', 'pickup_borough']).agg({
    'avg_duration': 'mean',
    'avg_price': 'mean',
    'avg_speed': 'mean',
    'trip_count': 'sum'
}).reset_index()

print("Zone Summary (top 10 by trips):")
zone_summary.sort_values('trip_count', ascending=False).head(10)

Zone Summary (top 10 by trips):


,pickup_zone,pickup_borough,avg_duration,avg_price,avg_speed,trip_count
67,East Harlem North,Manhattan,11.956948,19.650336,11.832357,33838
68,East Harlem South,Manhattan,11.428277,20.410944,12.997089,23550
38,Central Park,Manhattan,12.188212,22.587988,12.530807,9389
151,Morningside Heights,Manhattan,12.921507,22.942881,12.581465,7616
221,Upper East Side North,Manhattan,11.395921,21.308874,12.369295,7097
146,Midtown Center,Manhattan,13.486535,24.947116,10.964027,6712
87,Forest Hills,Queens,13.437052,21.985131,12.905866,6568
222,Upper East Side South,Manhattan,10.783862,20.062482,11.032014,6357
36,Central Harlem,Manhattan,12.283038,20.720716,12.292176,6188
114,JFK Airport,Queens,36.548827,70.113453,26.786380,5624


In [3]:
print("Step 1: Clustering Zones...")

# Use avg_duration, avg_price, avg_speed as clustering features
features = ['avg_duration', 'avg_price', 'avg_speed']
X_zone = zone_summary[features].fillna(0)

# Scale the features so KMeans works properly
scaler = StandardScaler()
X_zone_scaled = scaler.fit_transform(X_zone)

# Train KMeans with 4 clusters
zone_model = KMeans(n_clusters=4, random_state=42, n_init=10)
zone_summary['cluster_id'] = zone_model.fit_predict(X_zone_scaled)

# Show cluster distribution
print("\nCluster Distribution:")
print(zone_summary['cluster_id'].value_counts().sort_index())

# Show cluster averages to understand what each cluster represents
print("\nCluster Averages:")
zone_summary.groupby('cluster_id')[features + ['trip_count']].mean().round(2)

Step 1: Clustering Zones...

Cluster Distribution:
cluster_id
0     96
1    123
2     12
3     17
Name: count, dtype: int64

Cluster Averages:


,avg_duration,avg_price,avg_speed,trip_count
cluster_id,,,,
0,29.76,31.36,14.64,70.35
1,15.71,24.11,12.86,2119.29
2,33.21,73.23,24.35,823.50
3,22.71,39.70,25.56,17.82


In [4]:
print("Step 2: Applying zone clusters to hourly metrics...")

# Map cluster IDs from summary back to the hourly zone_metrics
cluster_map = zone_summary.set_index('pickup_zone')['cluster_id'].to_dict()
zone_metrics['cluster_id'] = zone_metrics['pickup_zone'].map(cluster_map)

print("\nStep 3: Clustering Corridors...")

# Use the same scaler to cluster corridor data
corridor_features = corridor_metrics[features].fillna(0)
corridor_scaled = scaler.transform(corridor_features)
corridor_metrics['cluster_id'] = zone_model.predict(corridor_scaled)

print("Zone cluster mapping applied:", len(cluster_map), "zones")
print("Corridor clusters assigned:", corridor_metrics['cluster_id'].value_counts().sort_index().to_dict())

Step 2: Applying zone clusters to hourly metrics...

Step 3: Clustering Corridors...
Zone cluster mapping applied: 248 zones
Corridor clusters assigned: {0: 13651, 1: 44512, 2: 7372, 3: 6912}


In [5]:
print("Step 4: Saving models and metadata...")

# Define what each cluster number means
cluster_metadata = {
    "zone_clusters": {
        "0": "High Demand Hub",
        "1": "Slow / Congested",
        "2": "Premium / Long Distance",
        "3": "Standard / Residential"
    },
    "corridor_clusters": {
        "0": "Reliable",
        "1": "Highly Volatile",
        "2": "Premium Corridor",
        "3": "Standard Route"
    }
}

# Save metadata as JSON
with open(models_folder + "cluster_metadata.json", "w") as f:
    json.dump(cluster_metadata, f, indent=2)

# Save the trained model and scaler
joblib.dump(zone_model, models_folder + "kmeans_zone.pkl")
joblib.dump(scaler, models_folder + "scaler_clustering.pkl")

# Save updated parquet files with cluster_id column
zone_metrics.to_parquet(data_folder + "zone_metrics.parquet", engine='fastparquet', index=False)
corridor_metrics.to_parquet(data_folder + "corridor_metrics.parquet", engine='fastparquet', index=False)

# Save zone summary for the Nearby Price feature
zone_summary.to_parquet(data_folder + "zone_summary.parquet", engine='fastparquet', index=False)

print("\nAll clustering work saved!")
print(f"  cluster_metadata.json: {len(cluster_metadata)} cluster types")
print(f"  kmeans_zone.pkl: KMeans model with {zone_model.n_clusters} clusters")
print(f"  scaler_clustering.pkl: StandardScaler")
print(f"  zone_metrics.parquet: {zone_metrics.shape}")
print(f"  corridor_metrics.parquet: {corridor_metrics.shape}")
print(f"  zone_summary.parquet: {zone_summary.shape}")

Step 4: Saving models and metadata...

All clustering work saved!
  cluster_metadata.json: 2 cluster types
  kmeans_zone.pkl: KMeans model with 4 clusters
  scaler_clustering.pkl: StandardScaler
  zone_metrics.parquet: (4524, 8)
  corridor_metrics.parquet: (72447, 7)
  zone_summary.parquet: (248, 7)
